# 📊 Análisis de Listings — Inside Airbnb Barcelona

Descarga los snapshots de listings directamente desde Inside Airbnb y analiza:
- Total de pisos
- Pisos con precio vs sin precio
- De los que tienen precio: cuántos están ocupados y cuántos no
- De los que **no** tienen precio: cuántos están ocupados y cuántos no

> **Proxy de ocupación:** un piso se considera *ocupado* si `number_of_reviews > 0` (tiene al menos una reseña), indicador estándar de actividad real en Airbnb.


In [ ]:
%pip install -q pandas requests tabulate


In [ ]:
import gzip
import io
import requests
import pandas as pd

# ── Configuración ──────────────────────────────────────────────────────────────
BASE_URL       = "https://data.insideairbnb.com/spain/catalonia/barcelona"
SNAPSHOT_DATES = ["2025-09-14", "2025-06-12"]

LISTING_COLS = [
    "id", "name", "neighbourhood_cleansed", "room_type",
    "accommodates", "bedrooms", "beds", "price",
    "minimum_nights", "number_of_reviews", "review_scores_rating",
    "instant_bookable",
]

# ── Descarga ───────────────────────────────────────────────────────────────────
def download_listings(snapshot_date: str) -> pd.DataFrame:
    url = f"{BASE_URL}/{snapshot_date}/data/listings.csv.gz"
    print(f"⬇️  Descargando snapshot {snapshot_date} ...")
    resp = requests.get(url, timeout=300)
    resp.raise_for_status()
    with gzip.open(io.BytesIO(resp.content), "rb") as f:
        df = pd.read_csv(f, low_memory=False)
    cols_disponibles = [c for c in LISTING_COLS if c in df.columns]
    df = df[cols_disponibles].copy()
    df["snapshot_date"] = snapshot_date
    print(f"   ✅ {len(df):,} filas descargadas")
    return df

# ── Descarga todos los snapshots y los une ─────────────────────────────────────
frames = [download_listings(d) for d in SNAPSHOT_DATES]
df_all = pd.concat(frames, ignore_index=True)

print(f"\n📦 Total de filas combinadas (todos los snapshots): {len(df_all):,}")
df_all.head(3)


In [ ]:
# ── Seleccionar IDs consistentes (tienen precio en TODOS los snapshots) ─────────
def find_consistent_ids(snapshots: list[str]) -> set:
    """
    Devuelve solo los IDs que tienen precio en TODOS los snapshots.
    """
    ids_por_snapshot = []
    for fecha in snapshots:
        sub = df_all[df_all["snapshot_date"] == fecha][["id", "price"]].copy()
        sub = sub[sub["price"].notna() & (sub["price"].astype(str).str.strip() != "")]
        ids_con_precio = set(sub["id"].dropna().astype(int).tolist())
        print(f"  Snapshot {fecha}: {len(ids_con_precio):,} listings con precio")
        ids_por_snapshot.append(ids_con_precio)

    consistent = ids_por_snapshot[0]
    for s in ids_por_snapshot[1:]:
        consistent = consistent.intersection(s)

    print(f"\n✅ Listings con precio en TODOS los snapshots: {len(consistent):,}")
    return consistent

consistent_ids = find_consistent_ids(SNAPSHOT_DATES)


In [ ]:
# ── Preparar DataFrame de análisis (un registro por listing único) ─────────────
# Usamos el snapshot más reciente para no duplicar pisos
df = df_all[df_all["snapshot_date"] == SNAPSHOT_DATES[0]].copy()

# Limpiar precio: quitar "$", "," y convertir a float
df["price_clean"] = (
    df["price"]
    .astype(str)
    .str.replace(r"[\$,]", "", regex=True)
    .str.strip()
)
df["tiene_precio"] = df["price_clean"].apply(
    lambda x: pd.notna(x) and x not in ("", "nan", "None")
)

# Ocupado: proxy = tiene al menos 1 reseña
df["ocupado"] = df["number_of_reviews"].fillna(0) > 0

# ── Tabla resumen ───────────────────────────────────────────────────────────────
total          = len(df)
con_precio     = df["tiene_precio"].sum()
sin_precio     = total - con_precio

cp_ocupados    = df[df["tiene_precio"]]["ocupado"].sum()
cp_no_ocupados = con_precio - cp_ocupados

sp_ocupados    = df[~df["tiene_precio"]]["ocupado"].sum()
sp_no_ocupados = sin_precio - sp_ocupados

print("=" * 62)
print(f"{'📋 RESUMEN GENERAL':^62}")
print("=" * 62)
print(f"  Total de pisos en el snapshot ({SNAPSHOT_DATES[0]}): {total:>8,}")
print(f"  ├─ Con precio:                                  {con_precio:>8,}  ({con_precio/total*100:.1f}%)")
print(f"  └─ Sin precio:                                  {sin_precio:>8,}  ({sin_precio/total*100:.1f}%)")
print()
print(f"{'💰 PISOS CON PRECIO':^62}")
print("-" * 62)
print(f"  ├─ Ocupados    (tienen reseñas):  {cp_ocupados:>8,}  ({cp_ocupados/con_precio*100:.1f}%)")
print(f"  └─ No ocupados (sin reseñas):     {cp_no_ocupados:>8,}  ({cp_no_ocupados/con_precio*100:.1f}%)")
print()
print(f"{'❌ PISOS SIN PRECIO':^62}")
print("-" * 62)
if sin_precio > 0:
    print(f"  ├─ Ocupados    (tienen reseñas):  {sp_ocupados:>8,}  ({sp_ocupados/sin_precio*100:.1f}%)")
    print(f"  └─ No ocupados (sin reseñas):     {sp_no_ocupados:>8,}  ({sp_no_ocupados/sin_precio*100:.1f}%)")
else:
    print("  (No hay pisos sin precio en este snapshot)")
print("=" * 62)


In [ ]:
from tabulate import tabulate

# ── Tabla bonita con tabulate ───────────────────────────────────────────────────
tabla = [
    ["Total pisos",                     total,          "100.0%"],
    ["  ├─ Con precio",                 con_precio,     f"{con_precio/total*100:.1f}%"],
    ["  │    ├─ Ocupados",              cp_ocupados,    f"{cp_ocupados/con_precio*100:.1f}%"],
    ["  │    └─ No ocupados",           cp_no_ocupados, f"{cp_no_ocupados/con_precio*100:.1f}%"],
    ["  └─ Sin precio",                 sin_precio,     f"{sin_precio/total*100:.1f}%"],
    ["       ├─ Ocupados",              sp_ocupados,    f"{sp_ocupados/sin_precio*100:.1f}%" if sin_precio > 0 else "—"],
    ["       └─ No ocupados",           sp_no_ocupados, f"{sp_no_ocupados/sin_precio*100:.1f}%" if sin_precio > 0 else "—"],
]

print(tabulate(tabla, headers=["Categoría", "Nº Pisos", "% sobre grupo padre"], tablefmt="rounded_outline"))
print(f"\n⚠️  Proxy de ocupación: number_of_reviews > 0  |  Snapshot: {SNAPSHOT_DATES[0]}")
